# 05 — Forecasting dataset and evaluation protocol

## Purpose

This notebook prepares the forecasting samples for our study of Gregorian, solar, lunar, and Vedic Panchang features. It brings the five feature families into one consistent experiment.

We prepare forecasts for **15, 30, and 60 minutes ahead** using the 5-minute SOLETE data. Each feature family uses the same observations and the same target values at a given forecast horizon.

Our **main research comparison is F3 versus F4**: does adding the five categorical Panchang groups improve forecasting beyond continuous solar and lunar astronomical information?

The notebook also defines the chronological development folds, the retrospective evaluation period, and the persistence baselines that will be used in the modelling notebooks.

## 1. Load the prepared SOLETE data

We use photovoltaic power as the forecasting target. Pressure, humidity, and wind direction are prepared using the cleaning rules established in Notebook 02.

The observations are indexed in UTC at 5-minute intervals. The recorded measurements are kept on their original timestamp grid.

In [1]:
from pathlib import Path
import gc
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (
        directory for directory in [Path.cwd(), *Path.cwd().parents]
        if (directory / "solete_dataset" / "SOLETE_Pombo_5min.h5").is_file()
        and (directory / "src" / "forecasting_protocol.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the SOLETE project.")
PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete
from src.preprocessing import build_cleaned_solete_features, build_gregorian_features
from src.astronomy_features import (
    build_lunar_f3_features,
    build_solar_f2_features,
    load_jpl_ephemeris,
    solete_astronomy_reference_times,
    tropical_longitudes_for_solete,
)
from src.vedic_calendar import (
    build_panchang_f4_raw,
    lahiri_ayanamsha_for_utc_times,
    sidereal_longitude_from_tropical,
    yoga_angle_from_tropical_longitudes,
)
from src.feature_families import (
    FEATURE_FAMILY_COUNTS,
    PANCHANG_INDICATOR_COUNT,
    encode_panchang_fixed_vocabulary,
)
from src.forecasting_protocol import (
    CV_FOLDS,
    F0_FEATURE_COLUMNS,
    FORECAST_HORIZONS,
    MODEL_AVAILABILITY_GAP,
    build_f0_history,
    build_forecast_targets,
    build_base_modelling_tables,
    make_rolling_origin_masks,
    make_terminal_refit_mask,
    terminal_holdout_mask,
    validate_rolling_origin_counts,
    validate_solete_5min_index,
)

DATA_FILE = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_5min.h5"
EPHEMERIS_FILE = PROJECT_ROOT / "ephemeris" / "de440s.bsp"
if not EPHEMERIS_FILE.is_file():
    raise FileNotFoundError(f"Expected astronomical ephemeris: {EPHEMERIS_FILE}")

raw = load_solete(DATA_FILE).sort_index()
cleaned = build_cleaned_solete_features(raw)
validate_solete_5min_index(cleaned)
assert len(cleaned) == 131_617
assert cleaned["P_Solar[kW]"].notna().all()

print("SOLETE observations:", len(cleaned))
print("Period:", cleaned.index.min(), "to", cleaned.index.max())
print("Target: P_Solar[kW]")

SOLETE observations: 131617
Period: 2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00
Target: P_Solar[kW]


## 2. Build historical predictors and forecast targets

A forecast is issued at time **t** and predicts PV power at **t + h**, where **h** is 15, 30, or 60 minutes.

The F0 predictors use nine measured variables at the issue time and the preceding 60 minutes, together with PV power at the issue time minus 24 hours. This produces **118 historical predictors**.

For each horizon, the target is the measured PV power at its forecast-valid time. We retain observations with complete historical predictors and an available future target.

In [2]:
f0_history = build_f0_history(cleaned)
targets = build_forecast_targets(cleaned)
base_tables = build_base_modelling_tables(
    cleaned,
    f0_history=f0_history,
    target_tables=targets,
)

assert f0_history.shape[1] == 118
assert tuple(FORECAST_HORIZONS) == (15, 30, 60)

population_rows = []
for horizon, base in base_tables.items():
    assert base.index.equals(pd.DatetimeIndex(base["issue_time"]))
    assert (base["valid_time"] - base["issue_time"] == pd.Timedelta(minutes=horizon)).all()
    assert base[list(F0_FEATURE_COLUMNS)].notna().all().all()
    assert base["target_P_Solar_kW"].notna().all()

    # Directly verify several future targets against the original measurements.
    for pos in np.linspace(0, len(base) - 1, 7, dtype=int):
        sample = base.iloc[pos]
        assert sample["target_P_Solar_kW"] == cleaned.loc[
            sample["valid_time"], "P_Solar[kW]"
        ]

    population_rows.append({
        "horizon_min": horizon,
        "samples": len(base),
        "first_issue_UTC": base.index.min(),
        "last_valid_UTC": base["valid_time"].max(),
    })

population_summary = pd.DataFrame(population_rows)
display(population_summary)
assert population_summary["samples"].tolist() == [131325, 131322, 131316]

,horizon_min,samples,first_issue_UTC,last_valid_UTC
0,15,131325,2018-06-02 00:00:00+00:00,2019-08-31 23:55:00+00:00
1,30,131322,2018-06-02 00:00:00+00:00,2019-08-31 23:55:00+00:00
2,60,131316,2018-06-02 00:00:00+00:00,2019-08-31 23:55:00+00:00


## 3. Calculate the deterministic feature blocks

The three additional feature blocks follow the definitions established in Notebooks 03 and 04:

- **F1:** 15 Gregorian calendar variables.
- **F2:** 5 solar-position variables.
- **F3:** 9 continuous lunar and Sun–Moon variables.
- **F4:** 97 categorical Panchang indicators.

These quantities are indexed by the **forecast-valid timestamp**. SOLETE solar and lunar astronomy uses the recorded timestamp plus 2 minutes as its astronomical reference time.

We also prepare four continuous sidereal-angle encodings for a secondary comparison of Panchang categories against a closer continuous reference.

In [3]:
timestamps = cleaned.index
f1_increment = build_gregorian_features(timestamps)
f2_increment = build_solar_f2_features(timestamps)

# Calculate the Sun–Moon geometry in batches to manage peak memory use.
ephemeris = load_jpl_ephemeris(EPHEMERIS_FILE)
BATCH_SIZE = 8192
lunar_parts = []
longitude_parts = []
for start in range(0, len(timestamps), BATCH_SIZE):
    batch = timestamps[start:start + BATCH_SIZE]
    lunar_parts.append(build_lunar_f3_features(batch, ephemeris))
    longitude_parts.append(tropical_longitudes_for_solete(batch, ephemeris))

f3_increment = pd.concat(lunar_parts)
tropical = pd.concat(longitude_parts)
del lunar_parts, longitude_parts
gc.collect()

reference_times = solete_astronomy_reference_times(timestamps)
ayanamsha = pd.Series(
    lahiri_ayanamsha_for_utc_times(reference_times),
    index=timestamps,
    name="lahiri_ayanamsha_deg",
)
panchang_raw = build_panchang_f4_raw(
    tropical["sun_tropical_longitude_deg"],
    tropical["moon_tropical_longitude_deg"],
    ayanamsha,
).add_prefix("valid_")
f4_increment = encode_panchang_fixed_vocabulary(panchang_raw)

moon_sidereal = sidereal_longitude_from_tropical(
    tropical["moon_tropical_longitude_deg"], ayanamsha
)
yoga_angle = yoga_angle_from_tropical_longitudes(
    tropical["sun_tropical_longitude_deg"],
    tropical["moon_tropical_longitude_deg"],
    ayanamsha,
)
continuous_sidereal = pd.DataFrame({
    "moon_sidereal_sin": np.sin(np.deg2rad(moon_sidereal)),
    "moon_sidereal_cos": np.cos(np.deg2rad(moon_sidereal)),
    "yoga_angle_sin": np.sin(np.deg2rad(yoga_angle)),
    "yoga_angle_cos": np.cos(np.deg2rad(yoga_angle)),
}, index=timestamps)

feature_blocks = {
    "Gregorian": f1_increment,
    "Solar": f2_increment,
    "Lunar": f3_increment,
    "Panchang": f4_increment,
    "Continuous sidereal": continuous_sidereal,
}
assert [len(x.columns) for x in feature_blocks.values()] == [15, 5, 9, 97, 4]
assert PANCHANG_INDICATOR_COUNT == 97
for block in feature_blocks.values():
    assert block.index.equals(timestamps)
    assert block.columns.is_unique
    assert block.notna().all().all()
assert (f4_increment.sum(axis=1) == 5).all()

print("Feature variables by block:")
display(pd.Series({name: len(table.columns) for name, table in feature_blocks.items()}, name="variables").to_frame())

Feature variables by block:


,variables
Gregorian,15
Solar,5
Lunar,9
Panchang,97
Continuous sidereal,4


## 4. Assemble the F0–F4 feature hierarchy

For every forecast horizon, the observational predictors come from the **issue time** and the calendar and astronomical predictors come from the **forecast-valid time**.

The feature sets are nested: each level retains all predictors from the preceding level. This allows us to compare the value of successive representations on the same forecasting examples.

The primary F3–F4 comparison uses **147 versus 244 predictors**. The secondary sidereal-parent comparison uses **151 versus 248 predictors**, with both sides sharing the same four additional continuous angles.

In [4]:
expected_counts = {"F0": 118, "F1": 133, "F2": 138, "F3": 147, "F4": 244}
assert FEATURE_FAMILY_COUNTS == expected_counts


def valid_block(table, valid_times, issue_times, *, prefix="valid_"):
    """Select deterministic features at valid time, indexed by issue time."""
    selected = table.loc[pd.DatetimeIndex(valid_times)].copy()
    selected.index = issue_times
    return selected.add_prefix(prefix)


feature_summary_rows = []
example_rows = []
for horizon, base in base_tables.items():
    issue_times = base.index
    valid_times = base["valid_time"]

    f0 = f0_history.loc[issue_times].copy()
    gregorian = valid_block(f1_increment, valid_times, issue_times)
    solar = valid_block(f2_increment, valid_times, issue_times)
    lunar = valid_block(f3_increment, valid_times, issue_times)
    panchang_indicators = valid_block(f4_increment, valid_times, issue_times, prefix="")
    sidereal = valid_block(continuous_sidereal, valid_times, issue_times)

    f1 = pd.concat([f0, gregorian], axis=1)
    f2 = pd.concat([f1, solar], axis=1)
    f3 = pd.concat([f2, lunar], axis=1)
    f4 = pd.concat([f3, panchang_indicators], axis=1)
    families = {"F0": f0, "F1": f1, "F2": f2, "F3": f3, "F4": f4}

    for name, matrix in families.items():
        assert matrix.index.equals(issue_times)
        assert matrix.shape == (len(base), expected_counts[name])
        assert matrix.columns.is_unique
        assert np.isfinite(matrix.to_numpy(dtype=np.float64)).all()
        feature_summary_rows.append({
            "horizon_min": horizon,
            "family": name,
            "samples": len(matrix),
            "predictors": matrix.shape[1],
        })

    for before, after in [("F0", "F1"), ("F1", "F2"), ("F2", "F3"), ("F3", "F4")]:
        assert list(families[after].columns[:families[before].shape[1]]) == list(families[before].columns)

    assert (panchang_indicators.sum(axis=1) == 5).all()
    assert pd.concat([f3, sidereal], axis=1).shape[1] == 151
    assert pd.concat([f4, sidereal], axis=1).shape[1] == 248

    pos = len(base) // 2
    issue = issue_times[pos]
    valid = valid_times.iloc[pos]
    assert f0.loc[issue, "P_Solar_kw_lag_0"] == cleaned.loc[issue, "P_Solar[kW]"]
    assert gregorian.loc[issue, "valid_utc_hour"] == valid.hour
    assert f2.loc[issue, "valid_solar_daylight"] == f2_increment.loc[valid, "solar_daylight"]
    assert f4.loc[issue, "valid_tithi_" + str(int(panchang_raw.loc[valid, "valid_tithi"]))] == 1

    example_rows.append({
        "horizon_min": horizon,
        "issue_UTC": issue,
        "valid_UTC": valid,
        "valid_Tithi": int(panchang_raw.loc[valid, "valid_tithi"]),
        "target_PV_kW": base.iloc[pos]["target_P_Solar_kW"],
    })

    # One horizon at a time keeps memory usage manageable.
    del f0, f1, f2, f3, f4, families, gregorian, solar, lunar, panchang_indicators, sidereal
    gc.collect()

feature_summary = pd.DataFrame(feature_summary_rows)
display(feature_summary.pivot(index="horizon_min", columns="family", values="predictors"))
print("Example aligned forecasting samples:")
display(pd.DataFrame(example_rows))

family,F0,F1,F2,F3,F4
horizon_min,,,,,
15,118,133,138,147,244
30,118,133,138,147,244
60,118,133,138,147,244


Example aligned forecasting samples:


,horizon_min,issue_UTC,valid_UTC,valid_Tithi,target_PV_kW
0,15,2019-01-15 23:50:00+00:00,2019-01-16 00:05:00+00:00,10,0.0
1,30,2019-01-15 23:45:00+00:00,2019-01-16 00:15:00+00:00,10,0.0
2,60,2019-01-15 23:30:00+00:00,2019-01-16 00:30:00+00:00,10,0.0


## 5. Define the chronological development folds

We use three expanding-window development folds. Training begins in June 2018, and each validation period follows the data used to fit the model.

A **60-minute availability gap** separates the final training target from the first validation forecast issue time. This ensures that every training label would already have been observed when the first validation prediction was issued.

The June–August 2019 period is reserved for **retrospective evaluation**. Its outcomes were examined in earlier research, so this period is described transparently as historically exposed rather than as a fresh independent test.

In [5]:
fold_rows = []
period_rows = []
for horizon, base in base_tables.items():
    valid_time = base["valid_time"]
    issue_time = base["issue_time"]
    folds = make_rolling_origin_masks(valid_time)
    assert len(folds) == 3
    assert MODEL_AVAILABILITY_GAP == pd.Timedelta(minutes=60)
    assert validate_rolling_origin_counts(valid_time, horizon) == tuple(
        (int(x["train_mask"].sum()), int(x["validation_mask"].sum())) for x in folds
    )

    for spec, masks in zip(CV_FOLDS, folds, strict=True):
        train, validation = masks["train_mask"], masks["validation_mask"]
        assert not (train & validation).any()
        assert valid_time.loc[train].max() < issue_time.loc[validation].min()
        fold_rows.append({
            "horizon_min": horizon,
            "fold": spec["fold"],
            "train_samples": int(train.sum()),
            "validation_samples": int(validation.sum()),
            "last_training_target_UTC": valid_time.loc[train].max(),
            "first_validation_issue_UTC": issue_time.loc[validation].min(),
        })

    retrospective = terminal_holdout_mask(valid_time)
    refit = make_terminal_refit_mask(valid_time)
    assert int(retrospective.sum()) == 26_496
    assert not (retrospective & refit).any()
    assert valid_time.loc[refit].max() < issue_time.loc[retrospective].min()
    assert valid_time.loc[folds[-1]["validation_mask"]].max() < issue_time.loc[retrospective].min()
    period_rows.append({
        "horizon_min": horizon,
        "development_refit_samples": int(refit.sum()),
        "retrospective_samples": int(retrospective.sum()),
        "evaluation_start_UTC": valid_time.loc[retrospective].min(),
        "evaluation_end_UTC": valid_time.loc[retrospective].max(),
    })

fold_summary = pd.DataFrame(fold_rows)
period_summary = pd.DataFrame(period_rows)
print("Chronological development folds:")
display(fold_summary)
print("Retrospective evaluation populations:")
display(period_summary)
assert period_summary["development_refit_samples"].tolist() == [104817, 104814, 104808]

Chronological development folds:


,horizon_min,fold,train_samples,validation_samples,last_training_target_UTC,first_validation_issue_UTC
0,15,1,52401,17856,2018-11-30 22:55:00+00:00,2018-11-30 23:45:00+00:00
1,15,2,70257,16992,2019-01-31 22:55:00+00:00,2019-01-31 23:45:00+00:00
2,15,3,87249,17556,2019-03-31 22:55:00+00:00,2019-03-31 23:45:00+00:00
3,30,1,52398,17856,2018-11-30 22:55:00+00:00,2018-11-30 23:30:00+00:00
4,30,2,70254,16992,2019-01-31 22:55:00+00:00,2019-01-31 23:30:00+00:00
5,30,3,87246,17556,2019-03-31 22:55:00+00:00,2019-03-31 23:30:00+00:00
6,60,1,52392,17856,2018-11-30 22:55:00+00:00,2018-11-30 23:00:00+00:00
7,60,2,70248,16992,2019-01-31 22:55:00+00:00,2019-01-31 23:00:00+00:00
8,60,3,87240,17556,2019-03-31 22:55:00+00:00,2019-03-31 23:00:00+00:00


Retrospective evaluation populations:


,horizon_min,development_refit_samples,retrospective_samples,evaluation_start_UTC,evaluation_end_UTC
0,15,104817,26496,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00
1,30,104814,26496,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00
2,60,104808,26496,2019-06-01 00:00:00+00:00,2019-08-31 23:55:00+00:00


## 6. Prepare the forecasting baselines

We will compare learned models with two simple methods:

- **Issue-time persistence:** use the PV power observed when the forecast is issued.
- **Previous-day persistence:** use PV power from 24 hours before the forecast-valid time.

Both baseline predictions are available at issue time. Their values are aligned with the exact rows used for F0–F4 comparisons.

In [6]:
baseline_rows = []
for horizon, base in base_tables.items():
    issue = base.index
    valid = pd.DatetimeIndex(base["valid_time"])
    issue_persistence = cleaned.loc[issue, "P_Solar[kW]"].to_numpy()
    previous_day_times = valid - pd.Timedelta(hours=24)
    previous_day = cleaned.loc[previous_day_times, "P_Solar[kW]"].to_numpy()
    target = base["target_P_Solar_kW"].to_numpy()

    assert len(target) == len(issue_persistence) == len(previous_day) == len(base)
    assert np.isfinite(issue_persistence).all()
    assert np.isfinite(previous_day).all()
    assert (previous_day_times <= issue).all()
    assert np.array_equal(target, cleaned.loc[valid, "P_Solar[kW]"].to_numpy())

    baseline_rows.append({
        "horizon_min": horizon,
        "paired_samples": len(target),
        "issue_persistence_available": len(issue_persistence),
        "previous_day_available": len(previous_day),
    })

display(pd.DataFrame(baseline_rows))

,horizon_min,paired_samples,issue_persistence_available,previous_day_available
0,15,131325,131325,131325
1,30,131322,131322,131322
2,60,131316,131316,131316


## 7. Comparisons for the modelling stage

We will use **mean absolute error (MAE)** as the primary forecasting metric and **root mean square error (RMSE)** as a complementary metric.

The nested comparisons examine the additional forecasting value of each block:

| Comparison | Question |
|---|---|
| F0 → F1 | Does Gregorian time help? |
| F1 → F2 | Does solar geometry help? |
| F2 → F3 | Does continuous lunar astronomy help? |
| **F3 → F4** | **Do categorical Panchang features help beyond continuous astronomy?** |

For the primary Panchang comparison, the signed effect is

\[
\Delta_\mathrm{Panchang} = \mathrm{MAE}(F4) - \mathrm{MAE}(F3).
\]

A value below zero means lower error with F4. Each comparison uses the same model architecture, training settings, horizon, and prediction timestamps on both sides.

As an additional representation analysis, we will compare **F3 + continuous sidereal angles** against **F3 + those same angles + Panchang indicators**. This tests categorical sector encodings against a more closely matched continuous astronomical parent.

In [7]:
comparison_plan = pd.DataFrame([
    ("Gregorian", "F0", "F1", 118, 133),
    ("Solar", "F1", "F2", 133, 138),
    ("Continuous lunar", "F2", "F3", 138, 147),
    ("Panchang — primary", "F3", "F4", 147, 244),
    ("Panchang — sidereal parent control", "F3 + 4 sidereal angles", "F4 + 4 sidereal angles", 151, 248),
], columns=["research_comparison", "reference", "augmented", "reference_features", "augmented_features"])
display(comparison_plan)

assert set(feature_summary["family"]) == set(expected_counts)
assert len(feature_summary) == 15
assert len(fold_summary) == 9
assert len(period_summary) == 3
assert (period_summary["retrospective_samples"] == 26496).all()
print("NOTEBOOK 05: FORECASTING DATASET CHECKS PASSED")

,research_comparison,reference,augmented,reference_features,augmented_features
0,Gregorian,F0,F1,118,133
1,Solar,F1,F2,133,138
2,Continuous lunar,F2,F3,138,147
3,Panchang — primary,F3,F4,147,244
4,Panchang — sidereal parent control,F3 + 4 sidereal angles,F4 + 4 sidereal angles,151,248


NOTEBOOK 05: FORECASTING DATASET CHECKS PASSED


## What we learned

- SOLETE provides matched forecasting samples for 15-, 30-, and 60-minute horizons.
- The five nested feature families contain **118, 133, 138, 147, and 244 predictors**.
- Historical measurements are aligned to the forecast issue time; deterministic calendar and astronomical features describe the forecast-valid time.
- Three chronological development folds use a 60-minute training-label availability gap.
- The retrospective SOLETE evaluation period contains **26,496 samples per horizon**.
- The F3–F4 comparison directly measures the forecasting response to categorical Panchang features, with a further continuous-sidereal-parent control available for sensitivity analysis.

The next notebook will use these samples and comparisons for model development.